# ResNet-18 on CIFAR-10: SGD, USAM, and randomized CV

**Research question:** Can randomized evaluation of the USAM correction reduce the time needed to obtain an acceptable model?

This standalone notebook first screens learning rates on validation data, freezes one configuration per method, and then benchmarks fresh seeds sequentially. It records the first checkpoint that confirms each validation-accuracy target at **two consecutive scheduled checks**, plus the final model. The primary full-run target is **92%**, with **90% and 94%** also reported. Change these before running the experiment if your definition of acceptable differs.

Only **SGD, USAM, and CV** are included. CV has fixed $p=1/2$. The experiment uses momentum SGD as the common base optimizer; no adaptive probability or adaptive-$h$ method is present.

**Start with `PROFILE='smoke'` and Run All.** Smoke uses a tiny synthetic dataset and the actual ResNet-18, so its accuracy/timing is meaningless. For the real experiment, set `PROFILE='full'`, confirm the settings, restart the kernel, and Run All. The full experiment was **not run before delivery**; this notebook supplies an experiment, not a claimed result.

All source code is embedded below. A cell writes one generated helper module inside `WORK_ROOT/embedded_code`; this is necessary for reliable multiprocessing workers from Jupyter on macOS/Windows. You do not need any external project files.

## Environment and user settings

Your existing **Python (mypapers)** kernel is suitable if it has compatible PyTorch and torchvision installations. Other environments need `torch torchvision numpy pandas scipy matplotlib ipykernel`. Install PyTorch/torchvision using the [official instructions](https://pytorch.org/get-started/locally/).

CPU workers parallelize data loading and augmentation within a single training run. Measured training runs are always sequential. Worker count can change real elapsed time; it is held identical across methods and saved in the protocol. Do not run competing training jobs or change worker count midway through a comparison. `NUM_WORKERS=0` is a portable fallback; 2 is the default.

Full defaults: batch 128, 45,000 training images, 5,000 validation images, five fresh benchmark seeds, and one CV coin replica per seed. Full CIFAR-10 uses the official training/test split; test images are decoded only in the optional final-scoring stage. The downloaded official archive contains both splits.

In [13]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import contextlib
import gc
import hashlib
import importlib
import itertools
import json
import math
import platform
import socket
import sys
import time
import uuid
import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib.pyplot as plt
from IPython.display import display


def choose_device(requested):
    if requested == 'auto':
        if torch.cuda.is_available(): return 'cuda'
        if torch.backends.mps.is_available(): return 'mps'
        return 'cpu'
    device = torch.device(requested)
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable.')
    if device.type == 'mps' and not torch.backends.mps.is_available():
        raise RuntimeError('MPS is unavailable.')
    return str(device)


def configuration(profile, requested_device, workers):
    if profile not in ('smoke', 'full'):
        raise ValueError('PROFILE must be smoke or full.')
    smoke = profile == 'smoke'
    chosen_device = choose_device(requested_device)
    return {
        'profile': profile, 'device': chosen_device,
        'data_source': 'synthetic' if smoke else 'cifar10', 'download': True,
        'methods': ['sgd', 'usam', 'cv'], 'batch_size': 4 if smoke else 128,
        'eval_batch_size': 20 if smoke else 256,
        'train_count': 40 if smoke else 45000, 'val_count': 20 if smoke else 5000,
        'test_count': 20 if smoke else 10000,
        'num_workers': workers, 'eval_num_workers': 0, 'prefetch_factor': 2,
        'pin_memory': chosen_device.startswith('cuda'), 'cpu_threads': 4, 'deterministic': True,
        'split_seed': 2026102801, 'subset_seed': 2026102802,
        'tuning_seeds': [41001],
        'benchmark_seeds': [51001] if smoke else [51001, 51002, 51003, 51004, 51005],
        'cv_replicas': 2 if smoke else 1,
        'lr_grid': [0.01, 0.02] if smoke else [0.03, 0.1, 0.2],
        'gamma_grid': [0.025] if smoke else [0.025, 0.1],
        'tuning_reference_epochs': 20, 'benchmark_reference_epochs': 100,
        'tuning_query_budget_override': 4 if smoke else None,
        'benchmark_query_budget_override': 8 if smoke else None,
        'validation_query_interval_override': 2 if smoke else None,
        'momentum': 0.9, 'weight_decay': 5e-4,
        'cosine_min_lr_fraction': 0.01, 'p': 0.5,
        'primary_target': 0.0 if smoke else 0.92,
        'targets': [0.0, 0.99] if smoke else [0.90, 0.92, 0.94],
        'target_confirmations': 2,
        'job_order_seed': 2026102803,
        'warmup': True, 'finite_checks': True,
    }


# USER SETTINGS. Smoke uses random synthetic images and is never a benchmark.
PROFILE = 'full'
DEVICE = 'auto'
NUM_WORKERS = 10  # 0 disables multiprocessing; use the SAME value for all methods.
WORK_ROOT = Path('cifar10_usam_cv_sgd')
DATA_DIR = WORK_ROOT / 'data'
OUTPUT_DIR = WORK_ROOT / f'results_{PROFILE}'
RESUME = True  # completed trials only; interrupted trials restart from initialization
RUN_TEST_SCORING = False  # enable only after selections and benchmark runs are complete
CFG = configuration(PROFILE, DEVICE, NUM_WORKERS)

# Edit targets/grids here BEFORE the first full run. A changed protocol needs a new OUTPUT_DIR.
# CFG['primary_target'] = 0.90
# CFG['gamma_grid'] = [0.01, 0.05, 0.1]

torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(CFG['deterministic'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = CFG['deterministic']
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
print(json.dumps(CFG, indent=2))
print('SMOKE: execution check only.' if PROFILE == 'smoke' else 'FULL: real CIFAR-10 tuning and benchmarking.')

{
  "profile": "full",
  "device": "mps",
  "data_source": "cifar10",
  "download": true,
  "methods": [
    "sgd",
    "usam",
    "cv"
  ],
  "batch_size": 128,
  "eval_batch_size": 256,
  "train_count": 45000,
  "val_count": 5000,
  "test_count": 10000,
  "num_workers": 10,
  "eval_num_workers": 0,
  "prefetch_factor": 2,
  "pin_memory": false,
  "cpu_threads": 4,
  "deterministic": true,
  "split_seed": 2026102801,
  "subset_seed": 2026102802,
  "tuning_seeds": [
    41001
  ],
  "benchmark_seeds": [
    51001,
    51002,
    51003,
    51004,
    51005
  ],
  "cv_replicas": 1,
  "lr_grid": [
    0.03,
    0.1,
    0.2
  ],
  "gamma_grid": [
    0.025,
    0.1
  ],
  "tuning_reference_epochs": 20,
  "benchmark_reference_epochs": 100,
  "tuning_query_budget_override": null,
  "benchmark_query_budget_override": null,
  "validation_query_interval_override": null,
  "momentum": 0.9,
  "weight_decay": 0.0005,
  "cosine_min_lr_fraction": 0.01,
  "p": 0.5,
  "primary_target": 0.92,
  "tar

## Mathematical updates and implementation choices

For the already-augmented minibatch $B$, compute $g=\nabla F_B(w)$ and, when required,
\[
q=\nabla F_B(w+\gamma g).
\]
The probe is detached and **unnormalized**; $\gamma$ is its coefficient, not a fixed radius. The same tensor of augmented images is reused for both queries.

| Method | Direction $D$ | Gradients per update |
|---|---|---:|
| SGD | $g$ | 1 |
| USAM | $q$ | 2 |
| CV | $g+2I(q-g)$, $I\sim\mathrm{Bernoulli}(1/2)$ | 1 or 2; mean 1.5 |

At a fixed model state and batch, $\mathbb E_I D_{CV}=q$ and $\operatorname{Cov}_I(D_{CV})=(q-g)(q-g)^\top$. Equality of conditional means does not imply identical trajectories or final accuracy.

All methods then apply the same base-optimizer update **once**:
\[
d=D+\omega w,\qquad v^+=\mu v+d,\qquad w^+=w-\eta_k v^+,
\]
with zero initial momentum, $\mu=0.9$, weight decay $\omega=5\times10^{-4}$, and no Nesterov momentum. Weight decay is excluded from the gradient used to construct the probe. Applying momentum separately to conceptual negative/positive substeps would implement a different algorithm.

The CIFAR ResNet-18 has a 3×3, stride-1 initial convolution, no initial max-pooling layer, and a 10-class output. It has 11,173,962 parameters. The unperturbed pass commits one BatchNorm running-stat update. The probe uses training-mode batch statistics, then restores **all** running buffers, including `num_batches_tracked`. It does not switch the probe to evaluation mode. Original parameters are restored by exact copies before the optimizer update. These details prevent the coin from changing BatchNorm bookkeeping. See the [SAM implementation's BatchNorm discussion](https://github.com/davda54/sam#training-tips).

There is no AMP, gradient clipping, gradient accumulation, early stopping on the test set, or silent failed-step retry. A nonfinite trajectory remains a recorded failure.

## Budgets, stepsize screening, and time to target

Let $M=\lceil N_{train}/\mathrm{batch\ size}\rceil$. Each method receives the same gradient-query budget
\[
G=2M E_{ref}.
\]
`E_ref` is a **USAM reference epoch count**, not the number of epochs every method trains. Full defaults use 20 reference epochs for each tuning trial and 100 for each benchmark trial. With 45,000 images and batch 128, these are 14,080 and 70,400 gradients. The benchmark gives approximately 200 SGD epochs, 100 USAM epochs, and 133.3 CV epochs. At most one query is left unused by CV; its coin is never changed to fill the budget.

The outer learning rate is scheduled by gradient evaluations already consumed, before the current coin:
\[
\eta_k=\eta_0\left[0.01+0.99\frac{1+\cos(\pi G_k/G)}2\right].
\]
The probe coefficient $\gamma$ is constant within a trial. This keeps decay tied to comparable gradient cost while allowing cheaper methods more updates.

The default basic grid is:

- SGD: $\eta_0\in\{0.03,0.1,0.2\}$, three candidates.
- USAM and CV: the same three rates, crossed with $\gamma\in\{0.025,0.1\}$, six candidates each.

One separate tuning seed is used by default, giving **15 short tuning trials**. This is screening, not proof of optimal hyperparameters. More tuning seeds can be configured. USAM/CV have more candidates than SGD; their additional search costs are explicitly reported.

Selection is fixed in advance: among candidates that complete successfully and confirm the primary target on every tuning seed, choose the smallest mean confirmation time. If none does, choose the highest mean final validation accuracy, breaking ties by lower loss. Freeze these choices before the fresh-seed benchmark. Full defaults give **15 benchmark trajectories**; increasing `cv_replicas` adds coin replicas with identical initialization and data streams.

Validation occurs after each common query milestone, normally every $2M$ queries. Confirmation times are observed on this grid, not exact continuous crossing times. A checkpoint is accepted only after two consecutive checks meet the target. All methods continue to their budget so final quality is also available. Target checkpoints remain valid if a later continuation fails.

The primary time starts **before model construction** and ends after the accepted checkpoint is saved and checksummed. It includes initialization, common discarded warmup, worker startup, batch waiting, transfers, optimizer work, safety checks, scheduled validation, and preceding checkpoint I/O. Shared download/preprocessing, hyperparameter search, and final test scoring are excluded and reported separately where measured. Component timings are also recorded. It is a measurement of this implementation and machine, not a theoretical gradient-count speedup.

Every planned run remains in the denominator. If a method misses a target in any planned run, its unconditional mean time to that target is suppressed; the attainment count is shown. A paired speedup is reported only when all planned runs of both methods reach the target. CV replicas are averaged within base seed before comparisons across seeds. Confidence intervals are exploratory and unadjusted.

## Embedded data code

Augmentations are random crop with zero padding of four pixels and horizontal flip. They are deterministic functions of the base seed, epoch, and original image index, independent of worker scheduling. All methods share the same update-indexed minibatch and augmentation prefixes. Normalization uses fixed CIFAR-10 constants, never test-derived estimates. Data come from the [official CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html) through torchvision's checked download.

In [14]:
DATA_SOURCE = r'''
"""Importable CIFAR-10 data helpers, embedded in the delivered notebook.

Training augmentations are keyed by (base seed, epoch, original sample index),
so changing DataLoader worker count does not change training examples. Workers
parallelize loading/augmentation only; the notebook runs training jobs serially.
"""
from pathlib import Path
import hashlib
import math
import random
import numpy as np
import torch
import torch.nn.functional as torch_f
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision.datasets import CIFAR10


CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
_MASK64 = (1 << 64) - 1


def mixed_seed(*values):
    """Stable integer mixing; independent of Python's process-randomized hash."""
    result = 0xA0761D6478BD642F
    for value in values:
        result = (result + int(value) + 0x9E3779B97F4A7C15) & _MASK64
        result = ((result ^ (result >> 30)) * 0xBF58476D1CE4E5B9) & _MASK64
        result = ((result ^ (result >> 27)) * 0x94D049BB133111EB) & _MASK64
        result ^= result >> 31
    return result & _MASK64


def worker_init(worker_id):
    """Bound intra-worker CPU use and seed incidental library randomness."""
    torch.set_num_threads(1)
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed)
    random.seed(seed)


def array_sha256(array):
    array = np.ascontiguousarray(array)
    return hashlib.sha256(memoryview(array).cast('B')).hexdigest()


def balanced_subset(labels, candidates, count, seed):
    candidates = np.asarray(candidates, dtype=np.int64)
    if int(count) != count or count <= 0 or count > len(candidates) or count % 10:
        raise ValueError('Subset counts must be positive multiples of 10 within the available split.')
    if count == len(candidates):
        return np.sort(candidates)
    rng = np.random.default_rng(int(seed))
    selected = []
    for label in range(10):
        members = candidates[labels[candidates] == label]
        if len(members) < count // 10:
            raise ValueError('Insufficient examples in a class for the balanced subset.')
        selected.extend(rng.choice(members, count // 10, replace=False))
    return np.sort(np.asarray(selected, dtype=np.int64))


def stratified_split(labels, validation_count, seed):
    if validation_count <= 0 or validation_count % 10:
        raise ValueError('Validation split count must be a positive multiple of 10.')
    rng = np.random.default_rng(int(seed))
    training, validation = [], []
    for label in range(10):
        members = np.flatnonzero(labels == label)
        if len(members) <= validation_count // 10:
            raise ValueError('Each class needs both training and validation examples.')
        rng.shuffle(members)
        validation.extend(members[:validation_count // 10])
        training.extend(members[validation_count // 10:])
    return np.sort(np.asarray(training, dtype=np.int64)), np.sort(np.asarray(validation, dtype=np.int64))


def synthetic_arrays(count, seed):
    if int(count) != count or count <= 0 or count % 10:
        raise ValueError('Synthetic example counts must be positive multiples of 10.')
    rng = np.random.default_rng(int(seed))
    images = rng.integers(0, 256, (count, 32, 32, 3), dtype=np.uint8)
    labels = np.tile(np.arange(10, dtype=np.int64), count // 10)
    # An independent order prevents labels from being grouped by class.
    permutation = rng.permutation(count)
    return images[permutation], labels[permutation]


def load_cifar_training(data_dir, cfg):
    """Load only official training examples; test decoding occurs separately.

    Torchvision downloads/checksums the official archive (which contains both
    splits), but CIFAR10(train=True) decodes only training batches. Synthetic
    data are explicitly marked and are for execution checks, never benchmarking.
    """
    source = cfg.get('data_source', 'cifar10')
    training_count = int(cfg.get('train_count', 45000))
    validation_count = int(cfg.get('val_count', 5000))
    split_seed = int(cfg.get('split_seed', 20261028))
    subset_seed = int(cfg.get('subset_seed', 20261029))
    if source == 'cifar10':
        dataset = CIFAR10(root=str(Path(data_dir)), train=True,
                          download=bool(cfg.get('download', True)))
        images = np.ascontiguousarray(dataset.data, dtype=np.uint8)
        labels = np.asarray(dataset.targets, dtype=np.int64)
        if images.shape != (50000, 32, 32, 3) or np.bincount(labels).tolist() != [5000]*10:
            raise ValueError('Unexpected official CIFAR-10 training structure.')
        base_validation_count = 5000
        archive = {'url': CIFAR10.url, 'md5': CIFAR10.tgz_md5,
                   'training_files': [{'name': name, 'md5': digest} for name, digest in CIFAR10.train_list]}
    elif source == 'synthetic':
        images, labels = synthetic_arrays(training_count + validation_count,
                                         mixed_seed(subset_seed, 0x534D4F4B45))
        base_validation_count = validation_count
        archive = None
    else:
        raise ValueError("data_source must be 'cifar10' or 'synthetic'.")
    train_pool, validation_pool = stratified_split(labels, base_validation_count, split_seed)
    train_indices = balanced_subset(labels, train_pool, training_count, subset_seed)
    validation_indices = balanced_subset(labels, validation_pool, validation_count, subset_seed + 1)
    if np.intersect1d(train_indices, validation_indices).size:
        raise AssertionError('Training and validation indices overlap.')
    metadata = {'source': source, 'train_count': len(train_indices),
                'validation_count': len(validation_indices), 'official_test_decoded': False,
                'split_seed': split_seed, 'subset_seed': subset_seed,
                'train_indices_sha256': array_sha256(train_indices),
                'validation_indices_sha256': array_sha256(validation_indices),
                'image_data_sha256': array_sha256(images), 'labels_sha256': array_sha256(labels),
                'normalization': {'mean': list(CIFAR_MEAN), 'std': list(CIFAR_STD),
                                  'source': 'fixed CIFAR-10 normalization constants; not estimated from test data'},
                'archive': archive}
    return {'images': images, 'labels': labels, 'train_indices': train_indices,
            'validation_indices': validation_indices, 'metadata': metadata}


def load_cifar_test(data_dir, cfg):
    """Explicit final-scoring call; never called from training or tuning helpers."""
    source = cfg.get('data_source', 'cifar10')
    count = int(cfg.get('test_count', 10000))
    seed = int(cfg.get('subset_seed', 20261029))
    if source == 'cifar10':
        dataset = CIFAR10(root=str(Path(data_dir)), train=False,
                          download=bool(cfg.get('download', True)))
        images = np.ascontiguousarray(dataset.data, dtype=np.uint8)
        labels = np.asarray(dataset.targets, dtype=np.int64)
        if images.shape != (10000, 32, 32, 3) or np.bincount(labels).tolist() != [1000]*10:
            raise ValueError('Unexpected official CIFAR-10 test structure.')
    elif source == 'synthetic':
        images, labels = synthetic_arrays(count, mixed_seed(seed, 0x54455354))
    else:
        raise ValueError("data_source must be 'cifar10' or 'synthetic'.")
    indices = balanced_subset(labels, np.arange(len(labels)), count, seed + 2)
    return {'images': images, 'labels': labels, 'test_indices': indices,
            'metadata': {'source': source, 'test_count': len(indices),
                         'indices_sha256': array_sha256(indices),
                         'image_data_sha256': array_sha256(images),
                         'labels_sha256': array_sha256(labels)}}


class CifarArrayDataset(Dataset):
    """Transforms are pure functions of explicit keys, including under spawn."""
    def __init__(self, images, labels, indices, augment=False, base_seed=0):
        images = np.ascontiguousarray(images, dtype=np.uint8)
        labels = np.ascontiguousarray(labels, dtype=np.int64)
        self.images = torch.from_numpy(images)
        self.labels = torch.from_numpy(labels)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.augment = bool(augment)
        self.base_seed = int(base_seed)
        self.mean = torch.tensor(CIFAR_MEAN, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(CIFAR_STD, dtype=torch.float32).view(3, 1, 1)
        if images.ndim != 4 or tuple(images.shape[1:]) != (32, 32, 3) or len(labels) != len(images):
            raise ValueError('Expected uint8 N x 32 x 32 x 3 images and N labels.')
        if self.indices.ndim != 1 or not len(self.indices) or self.indices.min() < 0 or self.indices.max() >= len(images):
            raise ValueError('Dataset indices must be nonempty valid row indices.')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, key):
        if isinstance(key, (tuple, list)):
            epoch, local_index = int(key[0]), int(key[1])
        else:
            epoch, local_index = 0, int(key)
        original_index = int(self.indices[local_index])
        image = self.images[original_index].permute(2, 0, 1)
        if self.augment:
            rng = np.random.default_rng(mixed_seed(self.base_seed, epoch, original_index, 0x415547))
            top, left = rng.integers(0, 9, size=2)
            image = torch_f.pad(image, (4, 4, 4, 4), mode='constant', value=0)
            image = image[:, int(top):int(top)+32, int(left):int(left)+32]
            if int(rng.integers(0, 2)):
                image = torch.flip(image, dims=(2,))
        image = image.to(dtype=torch.float32).div(255.)
        image = (image - self.mean) / self.std
        return image, self.labels[original_index]


class EpochBatchSampler(Sampler):
    """Yield (epoch,index) keys; retain ragged batches and reproducible prefixes."""
    def __init__(self, count, batch_size, total_updates, base_seed, start_update=0):
        values = (count, batch_size, total_updates, start_update)
        if any(int(v) != v for v in values) or min(count, batch_size, total_updates) <= 0 or start_update < 0:
            raise ValueError('Use positive integer counts and a nonnegative starting update.')
        self.count, self.batch_size = int(count), int(batch_size)
        self.total_updates, self.base_seed = int(total_updates), int(base_seed)
        self.start_update = int(start_update)
        self.batches_per_epoch = math.ceil(self.count / self.batch_size)

    def __len__(self):
        return self.total_updates

    def __iter__(self):
        first_epoch = self.start_update // self.batches_per_epoch
        first_batch = self.start_update % self.batches_per_epoch
        yielded = 0
        epoch = first_epoch
        while yielded < self.total_updates:
            rng = np.random.default_rng(mixed_seed(self.base_seed, epoch, 0x53414D504C45))
            order = rng.permutation(self.count)
            for batch in range(first_batch if epoch == first_epoch else 0, self.batches_per_epoch):
                start = batch * self.batch_size
                yield [(epoch, int(index)) for index in order[start:start+self.batch_size]]
                yielded += 1
                if yielded >= self.total_updates:
                    return
            epoch += 1


def loader_options(cfg, base_seed, evaluation=False):
    workers = cfg.get('eval_num_workers', cfg.get('num_workers', 0)) if evaluation else cfg.get('num_workers', 0)
    if int(workers) != workers or workers < 0:
        raise ValueError('num_workers must be a nonnegative integer.')
    workers = int(workers)
    generator = torch.Generator(device='cpu').manual_seed(mixed_seed(base_seed, 0x574F524B4552) % (2**63-1))
    options = {'num_workers': workers, 'pin_memory': bool(cfg.get('pin_memory', False)),
               'worker_init_fn': worker_init, 'generator': generator}
    if workers:
        prefetch = cfg.get('prefetch_factor', 2)
        if int(prefetch) != prefetch or prefetch < 1:
            raise ValueError('prefetch_factor must be a positive integer.')
        options.update(prefetch_factor=int(prefetch), persistent_workers=True,
                       multiprocessing_context='spawn')
    return options


def make_train_loader(data, cfg, base_seed, total_updates):
    dataset = CifarArrayDataset(data['images'], data['labels'], data['train_indices'],
                               augment=True, base_seed=base_seed)
    sampler = EpochBatchSampler(len(dataset), cfg['batch_size'], total_updates,
                                base_seed, start_update=0)
    return DataLoader(dataset, batch_sampler=sampler, **loader_options(cfg, base_seed))


def make_eval_loader(data, cfg, split='validation'):
    if split not in ('validation', 'train', 'test'):
        raise ValueError('Evaluation split must be validation, train, or test.')
    key = {'validation': 'validation_indices', 'train': 'train_indices', 'test': 'test_indices'}[split]
    dataset = CifarArrayDataset(data['images'], data['labels'], data[key], augment=False)
    return DataLoader(dataset, batch_size=cfg.get('eval_batch_size', 256), shuffle=False,
                      drop_last=False, **loader_options(cfg, 0x4556414C, evaluation=True))


def close_loader(loader):
    """Release persistent workers before another sequential timing job starts."""
    iterator = getattr(loader, '_iterator', None)
    if iterator is not None:
        shutdown = getattr(iterator, '_shutdown_workers', None)
        if shutdown is not None:
            shutdown()
        loader._iterator = None
'''

## Embedded model and optimizer code

In [15]:
ENGINE_SOURCE = r'''
"""First-order SGD, USAM, and unbiased CV updates for CIFAR ResNet-18.

The probe is constructed from the data-loss gradient only. Momentum and weight
decay are applied by the base SGD optimizer exactly once, after forming the
final direction. The module performs no data downloading or training at import.
"""
from contextlib import contextmanager
import math

import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import resnet18


METHODS = ("sgd", "usam", "cv")


def build_model(seed, device="cpu", dtype=torch.float32):
    """CIFAR ResNet-18: 3x3/stride-1 stem, no initial max pool, 10 outputs.

    Initialization is performed on CPU with an isolated RNG so every method
    receives exactly the same initialization for a given base seed.
    """
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(int(seed))
        model = resnet18(weights=None, num_classes=10)
        model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        nn.init.kaiming_normal_(model.conv1.weight, mode="fan_out", nonlinearity="relu")
        model.maxpool = nn.Identity()
    return model.to(device=device, dtype=dtype)


def synchronize_device(device):
    device = torch.device(device)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def _bn_snapshot(model):
    snapshots = []
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            for name in ("running_mean", "running_var", "num_batches_tracked"):
                value = getattr(module, name, None)
                if value is not None:
                    snapshots.append((value, value.detach().clone()))
    return snapshots


def _restore_buffers(snapshots):
    with torch.no_grad():
        for buffer, saved in snapshots:
            buffer.copy_(saved)


@contextmanager
def probe_batchnorm(model):
    """Use training-mode batch statistics without committing a second update.

    This restores running_mean, running_var AND num_batches_tracked after the
    probe, including when its forward/backward pass raises an exception. No
    module is switched to eval mode, and no BatchNorm momentum/flags are edited.
    """
    snapshots = _bn_snapshot(model)
    try:
        yield
    finally:
        _restore_buffers(snapshots)


def _all_finite(tensors):
    return bool(torch.stack([torch.isfinite(t).all() for t in tensors]).all().item())


def _global_norm(tensors):
    return torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(t) for t in tensors]))


def optimizer_step(model, optimizer, xb, yb, method, eta, gamma,
                   selected=None, check_finite=True, on_query=None):
    """Perform ONE logical update, with at most two gradients on the SAME batch.

    `on_query`, if supplied, is called immediately before every attempted
    loss-gradient query; the caller can count failed queries without retries.

    SGD uses D=g. USAM uses D=q, q=grad F_B(w+gamma*g). CV uses D=g on an
    unselected branch and D=2*q-g on a selected branch. The caller must draw an
    independent Bernoulli(1/2) coin before each CV update. gamma is an
    unnormalized coefficient, not a perturbation radius.

    The base optimizer must be torch.optim.SGD. Its momentum and weight decay
    act once on D at the restored, unperturbed weights. Gradient buffers never
    contain a derivative through the probe construction. On a probe exception,
    original parameters and pre-step BatchNorm buffers are restored; no base
    optimizer step is attempted. A nonfinite optimizer endpoint is reported as
    failure, not rolled back or retried.
    """
    if method not in METHODS:
        raise ValueError(f"Unknown method: {method}")
    if not isinstance(optimizer, torch.optim.SGD):
        raise TypeError("This experiment uses a torch.optim.SGD base optimizer.")
    if not math.isfinite(eta) or eta <= 0 or not math.isfinite(gamma) or gamma < 0:
        raise ValueError("Require finite eta>0 and gamma>=0.")
    if method == "cv" and not isinstance(selected, bool):
        raise ValueError("CV requires a Boolean, independently sampled fair coin.")
    if not model.training:
        raise ValueError("Call model.train() before a training update.")
    parameters = [p for p in model.parameters() if p.requires_grad]
    if not parameters:
        raise ValueError("The model has no trainable parameters.")
    optimized = [p for group in optimizer.param_groups for p in group["params"]]
    if len(optimized) != len(parameters) or {id(p) for p in optimized} != {id(p) for p in parameters}:
        raise ValueError("The base optimizer must contain every trainable parameter exactly once.")
    for group in optimizer.param_groups:
        group["lr"] = float(eta)
    if xb.device != parameters[0].device or yb.device != parameters[0].device:
        raise ValueError("Move the batch to the model device before optimizer_step.")

    queries = 0
    initial_bn = _bn_snapshot(model)
    saved_parameters = None

    def backward_query():
        nonlocal queries
        optimizer.zero_grad(set_to_none=True)
        queries += 1
        if on_query is not None:
            on_query()
        loss = F.cross_entropy(model(xb), yb, reduction="mean")
        loss.backward()
        if any(p.grad is None for p in parameters):
            raise ValueError("Every trainable parameter must contribute to the loss.")
        gradient = [p.grad.detach() for p in parameters]
        if check_finite and not _all_finite([loss.detach(), *gradient]):
            raise FloatingPointError("Nonfinite loss or gradient; update not applied.")
        return loss.detach(), gradient

    try:
        loss, g = backward_query()
        q = None
        expensive = method == "usam" or (method == "cv" and selected)
        if expensive:
            saved_parameters = [p.detach().clone() for p in parameters]
            try:
                with torch.no_grad():
                    for p, gp in zip(parameters, g):
                        p.add_(gp, alpha=float(gamma))
                if check_finite and not _all_finite(parameters):
                    raise FloatingPointError("Nonfinite perturbation point; update not applied.")
                with probe_batchnorm(model):
                    _, q = backward_query()
            finally:
                # Exact copies avoid accumulation of add/subtract roundoff.
                with torch.no_grad():
                    for p, saved in zip(parameters, saved_parameters):
                        p.copy_(saved)
        if method == "usam":
            direction = q
        elif method == "cv" and selected:
            direction = [2.0 * qp - gp for gp, qp in zip(g, q)]
        else:
            direction = g
        if check_finite and not _all_finite(direction):
            raise FloatingPointError("Nonfinite combined direction; update not applied.")
        diagnostic_values = [loss, _global_norm(g)]
        if q is not None:
            diagnostic_values += [_global_norm(q), _global_norm([qp-gp for gp, qp in zip(g, q)])]
        numbers = torch.stack(diagnostic_values).detach().cpu().tolist()
        if check_finite and not all(math.isfinite(v) for v in numbers):
            raise FloatingPointError("Nonfinite optimizer diagnostic; update not applied.")
        for p, direction_part in zip(parameters, direction):
            p.grad = direction_part.detach()
    except Exception:
        _restore_buffers(initial_bn)
        optimizer.zero_grad(set_to_none=True)
        raise

    # Weight decay and momentum are applied only here, at the restored weights.
    optimizer.step()
    if check_finite and not _all_finite(parameters):
        raise FloatingPointError("Nonfinite optimizer endpoint; stop this trajectory.")
    return {
        "queries": queries, "loss": float(numbers[0]),
        "gradient_norm": float(numbers[1]),
        "probe_gradient_norm": float(numbers[2]) if q is not None else None,
        "correction_norm": float(numbers[3]) if q is not None else None,
        "eta": float(eta), "gamma": float(gamma),
        "selected": bool(selected) if method == "cv" else None,
        "examples_queried": queries * len(yb),
    }


def evaluate_model(model, loader, device):
    """Evaluate unperturbed weights without changing BN buffers or training mode."""
    was_training = model.training
    model.eval()
    dtype = next(model.parameters()).dtype
    loss_sum = torch.zeros((), device=device, dtype=dtype)
    correct = torch.zeros((), device=device, dtype=torch.int64)
    count = 0
    try:
        with torch.inference_mode():
            for batch in loader:
                xb, yb = batch[:2]
                xb = xb.to(device=device, dtype=dtype, non_blocking=True)
                yb = yb.to(device=device, dtype=torch.long, non_blocking=True)
                scores = model(xb)
                loss_sum += F.cross_entropy(scores, yb, reduction="sum")
                correct += (scores.argmax(dim=1) == yb).sum()
                count += len(yb)
        if count == 0:
            raise ValueError("Evaluation loader is empty.")
        values = torch.stack([loss_sum / count, correct.to(dtype) / count]).cpu().tolist()
        if not all(math.isfinite(v) for v in values):
            raise FloatingPointError("Nonfinite evaluation metric.")
        return {"loss": float(values[0]), "accuracy": float(values[1]), "count": count}
    finally:
        model.train(was_training)
'''

## Embedded experiment runner

In [16]:
RUNNER_SOURCE = r'''
"""Sequential, validation-only tuning and fresh-seed time-to-target comparison."""
from pathlib import Path
import contextlib
import gc
import hashlib
import itertools
import json
import math
import os
import platform
import socket
import time
import uuid
import numpy as np
import torch
import torchvision

# The notebook prepends the engine and data helpers to this module's source.


def json_clean(value):
    if isinstance(value, dict): return {str(k): json_clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_clean(v) for v in value]
    if isinstance(value, Path): return str(value)
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if math.isfinite(float(value)) else None
    if isinstance(value, np.bool_): return bool(value)
    return value


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_clean(value), indent=2, allow_nan=False))
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''): digest.update(block)
    return digest.hexdigest()


def config_hash(value):
    return hashlib.sha256(json.dumps(json_clean(value), sort_keys=True).encode()).hexdigest()


def runtime_environment(cfg):
    device = torch.device(cfg['device'])
    hardware = torch.cuda.get_device_name(device) if device.type == 'cuda' else platform.machine()
    return {'python': platform.python_version(), 'torch': torch.__version__,
            'torchvision': torchvision.__version__, 'numpy': np.__version__,
            'platform': platform.platform(), 'device': cfg['device'], 'hardware': hardware,
            'cpu_threads': torch.get_num_threads(),
            'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
            'cudnn_benchmark': torch.backends.cudnn.benchmark,
            'cudnn_deterministic': torch.backends.cudnn.deterministic,
            'cudnn_allow_tf32': torch.backends.cudnn.allow_tf32,
            'cuda_allow_tf32': torch.backends.cuda.matmul.allow_tf32,
            'default_dtype': str(torch.get_default_dtype())}


def validate_config(cfg):
    if cfg['methods'] != ['sgd', 'usam', 'cv'] or cfg['p'] != .5:
        raise ValueError('This study implements SGD, USAM, and CV with p=1/2.')
    if cfg['primary_target'] not in cfg['targets'] or len(set(cfg['targets'])) != len(cfg['targets']):
        raise ValueError('Targets must be distinct and contain primary_target.')
    if not all(0 <= t <= 1 for t in cfg['targets']):
        raise ValueError('Accuracy targets are fractions in [0,1].')
    for key in ('batch_size', 'eval_batch_size', 'cpu_threads', 'prefetch_factor',
                'cv_replicas', 'target_confirmations', 'tuning_reference_epochs', 'benchmark_reference_epochs'):
        if int(cfg[key]) != cfg[key] or cfg[key] < 1: raise ValueError(f'Invalid {key}.')
    for key in ('num_workers', 'eval_num_workers'):
        if int(cfg[key]) != cfg[key] or cfg[key] < 0: raise ValueError(f'Invalid {key}.')
    for key in ('tuning_query_budget_override', 'benchmark_query_budget_override', 'validation_query_interval_override'):
        value = cfg[key]
        if value is not None and (not math.isfinite(value) or int(value) != value or value < 1):
            raise ValueError(f'{key} must be a positive integer or None.')
    if not cfg['lr_grid'] or not cfg['gamma_grid']:
        raise ValueError('Provide learning-rate and probe-coefficient candidates.')
    if not all(math.isfinite(x) and x > 0 for x in cfg['lr_grid']):
        raise ValueError('Learning rates must be finite and positive.')
    if not all(math.isfinite(x) and x >= 0 for x in cfg['gamma_grid']):
        raise ValueError('Probe coefficients must be finite and nonnegative.')
    if len(set(cfg['lr_grid'])) != len(cfg['lr_grid']) or len(set(cfg['gamma_grid'])) != len(cfg['gamma_grid']):
        raise ValueError('Grid values must be distinct.')
    if not (0 < cfg['cosine_min_lr_fraction'] <= 1 and 0 <= cfg['momentum'] < 1 and cfg['weight_decay'] >= 0):
        raise ValueError('Invalid optimizer or schedule parameters.')
    for key in ('tuning_seeds', 'benchmark_seeds'):
        if not cfg[key] or len(set(cfg[key])) != len(cfg[key]): raise ValueError('Use distinct nonempty seed lists.')
    if set(cfg['tuning_seeds']) & set(cfg['benchmark_seeds']):
        raise ValueError('Tuning and benchmark seeds must be disjoint.')


def phase_budget(data, cfg, phase):
    epoch_updates = math.ceil(len(data['train_indices']) / cfg['batch_size'])
    override = cfg[f'{phase}_query_budget_override']
    budget = int(override if override is not None else 2 * epoch_updates * cfg[f'{phase}_reference_epochs'])
    interval = int(cfg['validation_query_interval_override'] or 2 * epoch_updates)
    if budget < 2 or interval < 1: raise ValueError('Budgets must allow at least two queries.')
    return budget, min(interval, budget)


def learning_rate_at_queries(initial_lr, queries_done, query_budget, minimum_fraction):
    fraction = min(1., max(0., queries_done / query_budget))
    return initial_lr * (minimum_fraction + (1 - minimum_fraction) * .5 * (1 + math.cos(math.pi * fraction)))


@contextlib.contextmanager
def exclusive_study_lock(root):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    path = root / 'active_training.lock'
    payload = {'pid': os.getpid(), 'host': socket.gethostname(), 'created': time.time()}
    try:
        descriptor = os.open(path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError:
        raise RuntimeError(f'Another study may be running: {path}. Do not run measured jobs concurrently. '
                           'After a crashed kernel, remove this lock only after confirming that its process stopped.')
    with os.fdopen(descriptor, 'w') as stream: json.dump(payload, stream)
    try:
        yield
    finally:
        if path.exists() and json.loads(path.read_text()) == payload: path.unlink()


def prepare_protocol(data, cfg, output_dir, implementation_sha256, resume=True):
    validate_config(cfg)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    protocol = {'configuration': cfg, 'data': data['metadata'],
                'implementation_sha256': implementation_sha256,
                'environment': runtime_environment(cfg),
                'architecture': 'CIFAR ResNet-18; 3x3 stride1 stem; no initial maxpool; 10 outputs',
                'selection': 'fastest primary-target confirmation if every tuning seed attains; otherwise best mean final validation accuracy, then loss',
                'timing': 'wall time from model construction through checkpoint availability; excludes shared data preparation, tuning, and test scoring',
                'version': 1}
    path = output_dir / 'protocol.json'
    if path.exists():
        if json.loads(path.read_text()) != json_clean(protocol):
            raise ValueError('Configuration, data, code, or runtime changed. Use a new OUTPUT_DIR.')
        if not resume: raise ValueError('Results already exist. Set RESUME=True or use a new OUTPUT_DIR.')
    else:
        if list(output_dir.iterdir()): raise ValueError('Output directory is nonempty without a matching protocol.')
        atomic_json(path, protocol)
    return config_hash(protocol)


def save_checkpoint(model, destination):
    destination = Path(destination)
    temporary = destination.with_suffix('.tmp.pt')
    # State dictionaries contain only tensors. Never serialize a model object.
    state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
    torch.save(state, temporary)
    temporary.replace(destination)
    return sha256_file(destination)


def run_trial(data, cfg, phase, method, params, seed, replica, trial_dir, relative_path):
    """One sequential trial. Wall time includes setup, warmup, loading, validation and saving."""
    trial_dir = Path(trial_dir)
    attempt = 'attempt-' + uuid.uuid4().hex[:12]
    checkpoint_dir = trial_dir / attempt
    checkpoint_dir.mkdir(parents=True, exist_ok=False)
    budget, interval = phase_budget(data, cfg, phase)
    device = cfg['device']
    synchronize_device(device)
    started = time.perf_counter()
    model = build_model(100000 + int(seed), device)
    initial_hash = hashlib.sha256()
    for tensor in model.state_dict().values(): initial_hash.update(tensor.detach().cpu().numpy().tobytes())
    model.train()
    initial_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()} if cfg['warmup'] else None
    optimizer = torch.optim.SGD(model.parameters(), lr=params['lr'], momentum=cfg['momentum'], weight_decay=cfg['weight_decay'])
    if cfg['warmup']:
        # Common discarded two-gradient warmup, with no training/test data.
        warm_x = torch.zeros(cfg['batch_size'], 3, 32, 32, device=device)
        warm_y = torch.arange(cfg['batch_size'], device=device) % 10
        optimizer_step(model, optimizer, warm_x, warm_y, 'usam', .01, 0., check_finite=True)
        model.load_state_dict(initial_state)
        optimizer = torch.optim.SGD(model.parameters(), lr=params['lr'], momentum=cfg['momentum'], weight_decay=cfg['weight_decay'])
        del initial_state, warm_x, warm_y
    train_loader = make_train_loader(data, cfg, int(seed), total_updates=budget)
    val_loader = make_eval_loader(data, cfg, split='validation')
    synchronize_device(device)
    setup_seconds = time.perf_counter() - started
    training_seconds = validation_seconds = checkpoint_seconds = 0.
    queries = examples = updates = selected_count = 0
    next_validation = interval
    rng = np.random.default_rng(mixed_seed(seed, replica, 0x434F494E))
    target_states = {str(float(t)): {'reached': False} for t in cfg['targets']}
    consecutive = {key: 0 for key in target_states}
    history = []
    failure = None
    final = None
    last_metric = None
    last_diagnostic = None
    iterator = None
    try:
        # Iterator creation starts worker processes; charge it to measured training.
        block_started = time.perf_counter()
        iterator = iter(train_loader)
        synchronize_device(device)
        training_seconds += time.perf_counter() - block_started
        while budget - queries >= (1 if method == 'sgd' else 2):
            synchronize_device(device)
            block_started = time.perf_counter()
            try:
                xb, yb = next(iterator)
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                # Select the current rate BEFORE the coin; never change the coin to fill a budget.
                eta = learning_rate_at_queries(params['lr'], queries, budget, cfg['cosine_min_lr_fraction'])
                selected = bool(rng.random() < .5) if method == 'cv' else None
                selected_count += int(selected is True)
                def count_query():
                    nonlocal queries, examples
                    queries += 1
                    examples += len(yb)
                diagnostic = optimizer_step(model, optimizer, xb, yb, method, eta, params['gamma'],
                    selected=selected, check_finite=cfg['finite_checks'], on_query=count_query)
                updates += 1
                last_diagnostic = diagnostic
                synchronize_device(device)
                training_seconds += time.perf_counter() - block_started
            except FloatingPointError as error:
                synchronize_device(device)
                training_seconds += time.perf_counter() - block_started
                failure = str(error)
                break
            is_final = budget - queries < (1 if method == 'sgd' else 2)
            if queries >= next_validation or is_final:
                validation_started = time.perf_counter()
                try:
                    metric = evaluate_model(model, val_loader, device)
                except FloatingPointError as error:
                    failure = str(error)
                    synchronize_device(device)
                    validation_seconds += time.perf_counter() - validation_started
                    break
                synchronize_device(device)
                validation_seconds += time.perf_counter() - validation_started
                last_metric = metric
                entry = {'queries': queries, 'updates': updates,
                         'elapsed_seconds': time.perf_counter() - started,
                         'training_seconds': training_seconds, 'validation_seconds': validation_seconds,
                         'validation_accuracy': metric['accuracy'], 'validation_loss': metric['loss'],
                         'lr': eta, 'gamma': params['gamma'],
                         'gradient_norm': diagnostic['gradient_norm'],
                         'correction_norm': diagnostic['correction_norm']}
                history.append(entry)
                newly_reached = []
                for key, state in target_states.items():
                    if state['reached']: continue
                    consecutive[key] = consecutive[key] + 1 if metric['accuracy'] >= float(key) else 0
                    if consecutive[key] >= cfg['target_confirmations']: newly_reached.append(key)
                if newly_reached:
                    save_started = time.perf_counter()
                    checkpoint = checkpoint_dir / f'target-q{queries}.pt'
                    digest = save_checkpoint(model, checkpoint)
                    checkpoint_seconds += time.perf_counter() - save_started
                    elapsed = time.perf_counter() - started
                    for key in newly_reached:
                        target_states[key] = {'reached': True, 'confirmation_queries': queries,
                            'confirmation_update': updates, 'elapsed_seconds': elapsed,
                            'validation_accuracy': metric['accuracy'], 'validation_loss': metric['loss'],
                            'checkpoint': str(checkpoint.relative_to(trial_dir)), 'checkpoint_sha256': digest}
                while next_validation <= queries: next_validation += interval
        if failure is None:
            if last_metric is None: raise AssertionError('A completed trajectory must have a final validation score.')
            save_started = time.perf_counter()
            checkpoint = checkpoint_dir / 'final.pt'
            digest = save_checkpoint(model, checkpoint)
            checkpoint_seconds += time.perf_counter() - save_started
            final = {'validation_accuracy': last_metric['accuracy'], 'validation_loss': last_metric['loss'],
                     'checkpoint': str(checkpoint.relative_to(trial_dir)), 'checkpoint_sha256': digest,
                     'queries': queries, 'updates': updates}
        synchronize_device(device)
        elapsed = time.perf_counter() - started
    finally:
        close_loader(train_loader)
        close_loader(val_loader)
        del iterator, train_loader, val_loader, optimizer, model
        gc.collect()
        if torch.device(device).type == 'mps': torch.mps.empty_cache()
        elif torch.device(device).type == 'cuda': torch.cuda.empty_cache()
    return {'phase': phase, 'method': method, 'base_seed': int(seed), 'replica': int(replica),
            'params': params, 'status': 'failed' if failure else 'complete', 'failure': failure,
            'gradient_calls': queries, 'training_examples_queried': examples, 'updates': updates,
            'selected_count': selected_count, 'query_budget': budget,
            'setup_seconds': setup_seconds, 'training_seconds': training_seconds,
            'validation_seconds': validation_seconds, 'checkpoint_seconds': checkpoint_seconds,
            'elapsed_seconds': elapsed, 'history': history, 'targets': target_states, 'final': final,
            'initial_weights_sha256': initial_hash.hexdigest(),
            'trial_relative_path': str(relative_path), 'warmup_gradient_calls': 2 if cfg['warmup'] else 0,
            'last_diagnostic': last_diagnostic}


def verify_checkpoints(record, trial_dir):
    checked = set()
    for endpoint in [record.get('final'), *record['targets'].values()]:
        if not endpoint or not endpoint.get('checkpoint'): continue
        name = endpoint['checkpoint']
        if name not in checked:
            if sha256_file(Path(trial_dir) / name) != endpoint['checkpoint_sha256']:
                raise ValueError('Saved model checksum mismatch.')
            checked.add(name)


def execute_job(data, cfg, output_dir, protocol_id, phase, method, params, seed, replica):
    identifier = f'{method}-{config_hash(params)[:10]}-s{seed}-r{replica}'
    relative = Path(phase) / identifier
    trial_dir = Path(output_dir) / relative
    result_path = trial_dir / 'result.json'
    if result_path.exists():
        record = json.loads(result_path.read_text())
        expected = (phase, method, params, int(seed), int(replica), protocol_id)
        actual = tuple(record[k] for k in ('phase', 'method', 'params', 'base_seed', 'replica', 'protocol_id'))
        if actual != expected: raise ValueError('Cached trial identity mismatch.')
        verify_checkpoints(record, trial_dir)
        print('Reused', identifier, flush=True)
        return record
    record = run_trial(data, cfg, phase, method, params, seed, replica, trial_dir, relative)
    record['protocol_id'] = protocol_id
    atomic_json(result_path, record)
    score = record['final']['validation_accuracy'] if record['final'] else 'FAILED'
    print(phase, identifier, 'validation:', score, 'seconds:', round(record['elapsed_seconds'], 2), flush=True)
    return record


def select_parameters(tuning_records, cfg):
    selections, table = {}, []
    primary = str(float(cfg['primary_target']))
    for method in cfg['methods']:
        candidates = {}
        for record in tuning_records:
            if record['method'] == method:
                candidates.setdefault(config_hash(record['params']), []).append(record)
        eligible = []
        for candidate_id, records in candidates.items():
            assert len(records) == len(cfg['tuning_seeds'])
            failed = any(r['status'] != 'complete' for r in records)
            all_reached = all(r['targets'][primary]['reached'] for r in records)
            row = {'method': method, 'params': records[0]['params'], 'candidate_id': candidate_id,
                   'failed': failed, 'all_tuning_seeds_reached': all_reached,
                   'target_seconds': float(np.mean([r['targets'][primary]['elapsed_seconds'] for r in records])) if all_reached else None,
                   'validation_accuracy': float(np.mean([r['final']['validation_accuracy'] for r in records])) if not failed else None,
                   'validation_loss': float(np.mean([r['final']['validation_loss'] for r in records])) if not failed else None,
                   'total_tuning_seconds': sum(r['elapsed_seconds'] for r in records)}
            table.append(row)
            if not failed: eligible.append(row)
        if not eligible:
            raise RuntimeError(f'Every {method} tuning candidate failed. Inspect failures and start a new configuration.')
        reached = [row for row in eligible if row['all_tuning_seeds_reached']]
        if reached:
            chosen = min(reached, key=lambda r: (r['target_seconds'], -r['validation_accuracy'], r['validation_loss'], r['candidate_id']))
            rule = 'fastest target confirmation across tuning seeds'
        else:
            chosen = min(eligible, key=lambda r: (-r['validation_accuracy'], r['validation_loss'], r['candidate_id']))
            rule = 'target not attained by a complete candidate; best final validation accuracy, then loss'
        selections[method] = {'params': chosen['params'], 'selection_rule': rule}
    return selections, table


def run_experiment(data, cfg, output_dir, implementation_sha256, lock_root, resume=True):
    output_dir = Path(output_dir)
    with exclusive_study_lock(lock_root):
        protocol_id = prepare_protocol(data, cfg, output_dir, implementation_sha256, resume)
        jobs = []
        for method in cfg['methods']:
            gammas = [0.] if method == 'sgd' else cfg['gamma_grid']
            for lr, gamma, seed in itertools.product(cfg['lr_grid'], gammas, cfg['tuning_seeds']):
                jobs.append((method, {'lr': float(lr), 'gamma': float(gamma)}, int(seed), 0))
        rng = np.random.default_rng(cfg['job_order_seed'])
        tuning = [execute_job(data, cfg, output_dir, protocol_id, 'tuning', *jobs[i]) for i in rng.permutation(len(jobs))]
        selections, table = select_parameters(tuning, cfg)
        selection = {'protocol_id': protocol_id, 'selections': selections, 'candidates': table,
                     'tuning_seconds': sum(r['elapsed_seconds'] for r in tuning),
                     'tuning_gradient_calls': sum(r['gradient_calls'] for r in tuning),
                     'test_data_used': False}
        selection_path = output_dir / 'selection.json'
        if selection_path.exists() and json.loads(selection_path.read_text()) != json_clean(selection):
            raise ValueError('Frozen selection differs from the current completed tuning records.')
        atomic_json(selection_path, selection)
        print('Frozen settings:', json.dumps(selections, indent=2), flush=True)
        jobs = [(method, selections[method]['params'], int(seed), rep)
                for method in cfg['methods'] for seed in cfg['benchmark_seeds']
                for rep in range(cfg['cv_replicas'] if method == 'cv' else 1)]
        benchmark = [execute_job(data, cfg, output_dir, protocol_id, 'benchmark', *jobs[i]) for i in rng.permutation(len(jobs))]
        for seed in cfg['benchmark_seeds']:
            assert len({r['initial_weights_sha256'] for r in benchmark if r['base_seed'] == seed}) == 1
        atomic_json(output_dir / 'tuning_records.json', tuning)
        atomic_json(output_dir / 'benchmark_records.json', benchmark)
        atomic_json(output_dir / 'training_complete.json', {'protocol_id': protocol_id,
                    'selection_sha256': sha256_file(selection_path), 'benchmark_trials': len(benchmark)})
    return tuning, benchmark, selection


def score_test_checkpoints(records, data_dir, cfg, output_dir, implementation_sha256):
    import copy
    output_dir = Path(output_dir)
    protocol = json.loads((output_dir / 'protocol.json').read_text())
    complete = json.loads((output_dir / 'training_complete.json').read_text())
    if protocol['configuration'] != cfg or protocol['implementation_sha256'] != implementation_sha256:
        raise ValueError('Configuration or implementation changed before test scoring.')
    if protocol['environment'] != runtime_environment(cfg):
        raise ValueError('Runtime changed before test scoring.')
    if complete['protocol_id'] != config_hash(protocol) or complete['selection_sha256'] != sha256_file(output_dir / 'selection.json'):
        raise ValueError('The frozen selection or completed-study marker changed.')
    expected = len(cfg['benchmark_seeds']) * (2 + cfg['cv_replicas'])
    if len(records) != expected: raise ValueError('Benchmark results are incomplete.')
    selections = json.loads((output_dir / 'selection.json').read_text())['selections']
    planned = {(method, int(seed), rep) for method in cfg['methods'] for seed in cfg['benchmark_seeds']
               for rep in range(cfg['cv_replicas'] if method == 'cv' else 1)}
    actual = [(record['method'], record['base_seed'], record['replica']) for record in records]
    if len(set(actual)) != len(actual) or set(actual) != planned:
        raise ValueError('Test scoring requires the exact unique planned benchmark identities.')
    for record in records:
        if (record['phase'] != 'benchmark' or record['protocol_id'] != complete['protocol_id']
                or record['params'] != selections[record['method']]['params']):
            raise ValueError('Test scoring received tuning, stale, or mismatched records.')
    test_data = load_cifar_test(data_dir, cfg)
    loader = make_eval_loader(test_data, cfg, split='test')
    scored = copy.deepcopy(records)
    model = build_model(0, cfg['device'])
    try:
        for record in scored:
            trial_dir = output_dir / record['trial_relative_path']
            verify_checkpoints(record, trial_dir)
            cache = {}
            for endpoint in [record.get('final'), *record['targets'].values()]:
                if not endpoint or not endpoint.get('checkpoint'): continue
                name = endpoint['checkpoint']
                if name not in cache:
                    state = torch.load(trial_dir / name, map_location='cpu', weights_only=True)
                    model.load_state_dict(state)
                    cache[name] = evaluate_model(model, loader, cfg['device'])
                endpoint.update(test_accuracy=cache[name]['accuracy'], test_loss=cache[name]['loss'])
    finally:
        close_loader(loader)
        del model
    atomic_json(output_dir / 'test_scored_records.json', {'records': scored, 'test_metadata': test_data['metadata']})
    return scored
'''

## Embedded analysis code

In [17]:
ANALYSIS_SOURCE = r'''
"""Failure-aware analysis of saved SGD/USAM/CV experiment records.

No data loading, model evaluation, checkpoint selection, or training occurs in
this module. Validation thresholds and benchmark seeds come from the protocol.
"""
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from scipy.stats import t as student_t
import matplotlib.pyplot as plt


METHOD_COLORS = {"sgd": "#3b6fb6", "usam": "#dd8b2c", "cv": "#2b9b79"}


def _finite(value):
    return isinstance(value, (int, float, np.integer, np.floating)) and math.isfinite(float(value))


def _json_value(value):
    if isinstance(value, dict):
        return {str(k): _json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_value(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if math.isfinite(float(value)) else None
    if isinstance(value, Path):
        return str(value)
    return value


def _benchmark_records(records, cfg):
    """Enumerate planned records explicitly, including any missing trajectories."""
    observed = {}
    methods = list(cfg["methods"])
    seeds = [int(s) for s in cfg["benchmark_seeds"]]
    if len(set(methods)) != len(methods) or len(set(seeds)) != len(seeds):
        raise ValueError("Method names and benchmark seeds must be unique.")
    for record in records:
        if record.get("phase") != "benchmark":
            continue
        key = (record["method"], int(record["base_seed"]), int(record.get("replica", 0)))
        if key in observed:
            raise ValueError(f"Duplicate benchmark trajectory: {key}")
        observed[key] = record
    planned = []
    expected = set()
    for method in methods:
        replicas = int(cfg["cv_replicas"]) if method == "cv" else 1
        if replicas < 1:
            raise ValueError("At least one replica is required.")
        for seed in seeds:
            for replica in range(replicas):
                key = (method, seed, replica)
                expected.add(key)
                planned.append(observed.get(key, {
                    "phase": "benchmark", "method": method, "base_seed": seed,
                    "replica": replica, "status": "missing",
                    "failure": "Planned benchmark record is missing.",
                    "targets": {}, "final": None, "history": [],
                }))
    unexpected = set(observed) - expected
    if unexpected:
        raise ValueError(f"Benchmark records outside the protocol: {sorted(unexpected)}")
    return planned


def _final_valid(record):
    final = record.get("final")
    return (record.get("status") == "complete" and isinstance(final, dict)
            and _finite(final.get("validation_accuracy"))
            and 0 <= final["validation_accuracy"] <= 1
            and _finite(final.get("validation_loss")))


def _target_entry(record, threshold):
    targets = record.get("targets") or {}
    entry = targets.get(str(float(threshold)))
    if entry is None:
        # Accommodate equivalent JSON spellings such as "0" and "0.0".
        for key, value in targets.items():
            try:
                matches = float(key) == float(threshold)
            except (ValueError, TypeError):
                matches = False
            if matches:
                entry = value
                break
    return entry if isinstance(entry, dict) else {"reached": False}


def _target_valid(entry, threshold):
    return (entry.get("reached") is True
            and _finite(entry.get("elapsed_seconds")) and entry["elapsed_seconds"] >= 0
            and _finite(entry.get("validation_accuracy"))
            and entry["validation_accuracy"] + 1e-12 >= float(threshold)
            and _finite(entry.get("validation_loss")))


def _base_means(records, getter):
    grouped = {}
    for record in records:
        value = getter(record)
        if not _finite(value):
            return None
        grouped.setdefault(int(record["base_seed"]), []).append(float(value))
    return {seed: float(np.mean(values)) for seed, values in grouped.items()}


def _mean_metric(records, getter, allowed):
    if not allowed:
        return np.nan
    values = _base_means(records, getter)
    return float(np.mean(list(values.values()))) if values else np.nan


def _confidence(values):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean()) if len(values) else np.nan
    if len(values) < 2:
        return mean, np.nan, np.nan
    halfwidth = float(student_t.ppf(.975, len(values) - 1) * values.std(ddof=1) / math.sqrt(len(values)))
    return mean, mean - halfwidth, mean + halfwidth


def summarize_results(records, cfg, output_dir):
    """Return and save final, target, paired, failure, and tuning-cost tables.

    Performance means at the final endpoint require every planned trajectory to
    finish. Target-time means require every planned trajectory to attain that
    target. Valid earlier target checkpoints survive a later numerical failure.
    CV replicas are averaged within each initialization seed before means and
    paired t intervals across seeds. Intervals are exploratory and unadjusted.
    """
    records = list(records)
    benchmark = _benchmark_records(records, cfg)
    methods = list(cfg["methods"])
    thresholds = [float(t) for t in cfg["targets"]]
    if len(set(thresholds)) != len(thresholds):
        raise ValueError("Targets must be distinct.")
    grouped = {method: [r for r in benchmark if r["method"] == method] for method in methods}
    final_rows, target_rows, failure_rows, paired_rows, cost_rows = [], [], [], [], []

    for method, group in grouped.items():
        all_complete = bool(group) and all(_final_valid(r) for r in group)
        final = {
            "method": method, "planned_runs": len(group),
            "base_seeds": len({r["base_seed"] for r in group}),
            "completed_runs": sum(_final_valid(r) for r in group),
            "failed_or_missing_runs": sum(not _final_valid(r) for r in group),
            "missing_runs": sum(r.get("status") == "missing" for r in group),
            "performance_status": "complete" if all_complete else "suppressed_incomplete_group",
        }
        for metric in ("validation_accuracy", "validation_loss", "test_accuracy", "test_loss"):
            final[metric] = _mean_metric(group, lambda r, m=metric: (r.get("final") or {}).get(m), all_complete)
        for metric in ("elapsed_seconds", "setup_seconds", "training_seconds", "validation_seconds",
                       "checkpoint_seconds", "gradient_calls", "updates", "query_budget"):
            final[metric] = _mean_metric(group, lambda r, m=metric: r.get(m), all_complete)
        final_rows.append(final)

        for threshold in thresholds:
            reached = [_target_valid(_target_entry(r, threshold), threshold) for r in group]
            all_reached = bool(group) and all(reached)
            target = {
                "method": method, "target_accuracy": threshold,
                "planned_runs": len(group), "base_seeds": len({r["base_seed"] for r in group}),
                "reached_runs": sum(reached), "unreached_runs": len(group) - sum(reached),
                "attainment_fraction": sum(reached) / len(group) if group else np.nan,
                "trajectory_failed_or_missing_runs": sum(not _final_valid(r) for r in group),
                "time_status": "all_reached" if all_reached else "suppressed_not_all_reached",
            }
            for source, destination in (("elapsed_seconds", "time_to_target_seconds"),
                                        ("confirmation_queries", "queries_to_target"),
                                        ("confirmation_update", "updates_to_target"),
                                        ("validation_accuracy", "validation_accuracy"),
                                        ("validation_loss", "validation_loss"),
                                        ("test_accuracy", "test_accuracy"), ("test_loss", "test_loss")):
                target[destination] = _mean_metric(group,
                    lambda r, key=source, t=threshold: _target_entry(r, t).get(key), all_reached)
            if all_reached:
                times = _base_means(group, lambda r, t=threshold: _target_entry(r, t)["elapsed_seconds"])
                _, low, high = _confidence(list(times.values()))
            else:
                low = high = np.nan
            target["time_exploratory_ci_low"] = low
            target["time_exploratory_ci_high"] = high
            target_rows.append(target)

    for record in benchmark + [r for r in records if r.get("phase") == "tuning"]:
        if _final_valid(record):
            continue
        failure_rows.append({
            "phase": record.get("phase"), "method": record.get("method"),
            "base_seed": record.get("base_seed"), "replica": record.get("replica", 0),
            "status": record.get("status", "missing"),
            "failure": record.get("failure") or "Final endpoint is missing or invalid.",
            "gradient_calls": record.get("gradient_calls"), "updates": record.get("updates"),
            "elapsed_seconds": record.get("elapsed_seconds"),
            "trial_relative_path": record.get("trial_relative_path"),
        })

    for threshold in thresholds:
        for reference in ("usam", "sgd"):
            if "cv" not in grouped or reference not in grouped:
                continue
            candidate, baseline = grouped["cv"], grouped[reference]
            expected_seeds = set(int(s) for s in cfg["benchmark_seeds"])
            valid_seeds = []
            for seed in sorted(expected_seeds):
                selected = [r for r in candidate + baseline if int(r["base_seed"]) == seed]
                if selected and all(_target_valid(_target_entry(r, threshold), threshold) for r in selected):
                    valid_seeds.append(seed)
            all_reached = len(valid_seeds) == len(expected_seeds) and bool(expected_seeds)
            row = {
                "method": "cv", "reference": reference, "target_accuracy": threshold,
                "planned_base_seeds": len(expected_seeds), "complete_base_pairs": len(valid_seeds),
                "status": "all_reached" if all_reached else "suppressed_not_all_pairs_reached",
                "mean_cv_seconds": np.nan, "mean_reference_seconds": np.nan,
                "mean_cv_over_reference_time": np.nan, "mean_time_saving_fraction": np.nan,
                "saving_exploratory_ci_low": np.nan, "saving_exploratory_ci_high": np.nan,
                "per_base_time_saving_fractions": [],
            }
            if all_reached:
                cv_times = _base_means(candidate, lambda r, t=threshold: _target_entry(r, t)["elapsed_seconds"])
                ref_times = _base_means(baseline, lambda r, t=threshold: _target_entry(r, t)["elapsed_seconds"])
                assert set(cv_times) == set(ref_times) == expected_seeds
                # Zero elapsed times can arise in synthetic fixtures, but are
                # not usable denominators for a time-ratio claim.
                if all(ref_times[s] > 0 for s in expected_seeds):
                    ratios = [cv_times[s] / ref_times[s] for s in sorted(expected_seeds)]
                    savings = [1. - ratio for ratio in ratios]
                    mean, low, high = _confidence(savings)
                    row.update(mean_cv_seconds=float(np.mean(list(cv_times.values()))),
                               mean_reference_seconds=float(np.mean(list(ref_times.values()))),
                               mean_cv_over_reference_time=float(np.mean(ratios)),
                               mean_time_saving_fraction=mean,
                               saving_exploratory_ci_low=low, saving_exploratory_ci_high=high,
                               per_base_time_saving_fractions=savings)
                else:
                    row["status"] = "suppressed_nonpositive_reference_time"
            paired_rows.append(row)

    tuning = [r for r in records if r.get("phase") == "tuning"]
    for method in methods:
        group = [r for r in tuning if r.get("method") == method]
        if not group:
            continue
        row = {
            "method": method, "trials": len(group),
            "candidate_settings": len({json.dumps(r.get("params", {}), sort_keys=True) for r in group}),
            "failed_trials": sum(not _final_valid(r) for r in group),
        }
        for metric in ("elapsed_seconds", "setup_seconds", "training_seconds", "validation_seconds",
                       "checkpoint_seconds", "gradient_calls", "updates"):
            values = [r.get(metric) for r in group]
            row["total_" + metric] = sum(float(v) for v in values) if all(_finite(v) for v in values) else np.nan
        cost_rows.append(row)

    failure_columns = ["phase", "method", "base_seed", "replica", "status", "failure",
                       "gradient_calls", "updates", "elapsed_seconds", "trial_relative_path"]
    result = {
        "final_summary": pd.DataFrame(final_rows),
        "target_summary": pd.DataFrame(target_rows),
        "paired_comparisons": pd.DataFrame(paired_rows),
        "failure_table": pd.DataFrame(failure_rows, columns=failure_columns),
        "tuning_costs": pd.DataFrame(cost_rows),
    }
    folder = Path(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    for name, table in result.items():
        table.to_csv(folder / (name + ".csv"), index=False)
    payload = {
        "interpretation": {
            "metrics": "Accuracy columns are fractions, times are seconds, and counts are actual recorded queries/updates.",
            "final_means": "Suppressed if any planned final endpoint failed or is missing.",
            "target_means": "Suppressed unless every planned run attains the target; valid targets survive later failures.",
            "replicas": "Average CV replicas within each base seed before averaging seeds or forming paired intervals.",
            "time_ratios": "Mean of per-base CV/reference time ratios, not the ratio of group mean times.",
            "intervals": "Exploratory, unadjusted paired Student-t intervals across base seeds; unavailable for one seed.",
            "clock": "Uses driver elapsed_seconds, including model setup/warmup, loading, training, validation, and checkpoint saving. Setup is a component, not an additional cost to add to elapsed. Tuning is reported separately.",
            "smoke": "Synthetic/smoke runs validate execution only and provide no performance evidence.",
        },
        **{name: table.to_dict(orient="records") for name, table in result.items()},
    }
    (folder / "analysis.json").write_text(json.dumps(_json_value(payload), indent=2, allow_nan=False))
    return result


def plot_results(records, cfg, output_dir):
    """Show trajectories and censor-aware attainment/time at the primary target."""
    records = list(records)
    planned = _benchmark_records(records, cfg)
    methods = list(cfg["methods"])
    target = float(cfg["primary_target"])
    if target not in [float(t) for t in cfg["targets"]]:
        raise ValueError("primary_target must appear in targets.")
    figure, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    seen = set()
    for record in planned:
        method = record["method"]
        valid = [h for h in record.get("history", [])
                 if _finite(h.get("elapsed_seconds")) and _finite(h.get("queries"))
                 and _finite(h.get("validation_accuracy"))]
        if not valid:
            continue
        label = method.upper() if method not in seen else None
        seen.add(method)
        style = "--" if not _final_valid(record) else "-"
        for axis, field in ((axes[0, 0], "elapsed_seconds"), (axes[0, 1], "queries")):
            axis.plot([h[field] for h in valid], [100*h["validation_accuracy"] for h in valid],
                      color=METHOD_COLORS.get(method), alpha=.55, linestyle=style,
                      marker="o", markersize=2.5, label=label)
    axes[0, 0].set(xlabel="Elapsed time including setup and validation (seconds)", ylabel="Validation accuracy (%)")
    axes[0, 1].set(xlabel="Gradient evaluations", ylabel="Validation accuracy (%)")
    for axis in axes[0]:
        axis.axhline(100*target, color="black", linestyle=":", linewidth=1,
                     label=f"Target {100*target:g}%")
        axis.legend(fontsize=8)
        axis.grid(alpha=.2)

    complete_means = []
    for x, method in enumerate(methods):
        group = [r for r in planned if r["method"] == method]
        entries = [_target_entry(r, target) for r in group]
        count = sum(_target_valid(e, target) for e in entries)
        fraction = count / len(group) if group else 0.
        axes[1, 0].bar(x, fraction, color=METHOD_COLORS.get(method))
        axes[1, 0].text(x, min(1.04, fraction+.035), f"{count}/{len(group)}", ha="center", fontsize=9)
        if group and count == len(group):
            times = _base_means(group, lambda r: _target_entry(r, target)["elapsed_seconds"])
            mean, low, high = _confidence(list(times.values()))
            error = max(0., high-mean) if _finite(high) else None
            axes[1, 1].bar(x, mean, color=METHOD_COLORS.get(method), yerr=error, capsize=4)
            axes[1, 1].text(x, mean, f"{mean:.1f}s", ha="center", va="bottom", fontsize=9)
            complete_means.append(mean + (error or 0.))
        else:
            axes[1, 1].text(x, .03, f"Not all reached\n({count}/{len(group)})",
                            transform=axes[1, 1].get_xaxis_transform(), ha="center", fontsize=9)
    labels = [m.upper() for m in methods]
    axes[1, 0].set(xticks=range(len(methods)), xticklabels=labels, ylim=(0, 1.15),
                   ylabel="Fraction of planned runs attaining target", title=f"Target: {100*target:g}% validation accuracy")
    axes[1, 1].set(xticks=range(len(methods)), xticklabels=labels,
                   ylabel="Time to confirmed target (seconds)",
                   title="Time means require every planned run to reach target")
    axes[1, 1].set_ylim(0, max(complete_means, default=1.) * 1.2 or 1.)
    for axis in axes[1]:
        axis.grid(axis="y", alpha=.2)
        axis.set_axisbelow(True)
    if cfg.get("profile") == "smoke":
        figure.suptitle("SMOKE CHECK — not a performance experiment", fontsize=12)
    folder = Path(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    figure.savefig(folder / "comparison.png", dpi=160)
    return figure
'''

## Embedded correctness checks

These use small synthetic examples. They test the update equations independently, one momentum/decay application, probe restoration, BatchNorm buffers, query counting, failures, and worker-independent data. They also exercise the actual ResNet-18. They do not estimate CIFAR-10 performance.

In [18]:
CHECK_SOURCE = r'''
"""Synthetic mathematical/implementation checks; no CIFAR training is run."""
import copy
import json
import math
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F



class TinyBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.first = nn.Linear(4, 5)
        self.bn = nn.BatchNorm1d(5)
        self.last = nn.Linear(5, 3)
        self.seen_batches = []

    def forward(self, x):
        self.seen_batches.append(id(x))
        return self.last(torch.tanh(self.bn(self.first(x))))


def run_engine_checks(real_resnet=True):
    checks = []
    old_threads = torch.get_num_threads()
    torch.set_num_threads(min(2, old_threads))
    try:
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(8501)
            initial = TinyBN().double().train()
        generator = torch.Generator().manual_seed(8502)
        xb = torch.randn(6, 4, generator=generator, dtype=torch.float64)
        yb = torch.tensor([0, 1, 2, 2, 1, 0])
        eta, gamma, momentum, decay = .037, .13, .9, .002
        original_values = [p.detach().clone() for p in initial.parameters()]
        old_momenta = [torch.full_like(p, .017 * (j + 1)) for j, p in enumerate(initial.parameters())]

        def make_optimizer(model, with_momentum=True):
            optimizer = torch.optim.SGD(model.parameters(), lr=eta,
                momentum=momentum if with_momentum else 0., weight_decay=decay)
            if with_momentum:
                for p, old in zip(model.parameters(), old_momenta):
                    optimizer.state[p]['momentum_buffer'] = old.clone()
            return optimizer

        # Independent reference: autograd.grad at two separate model copies;
        # manually apply the SGD momentum/decay equation, with one BN update.
        def independent_reference(method, selected, probe_coefficient=gamma):
            base = copy.deepcopy(initial)
            params = list(base.parameters())
            loss = F.cross_entropy(base(xb), yb)
            g = list(torch.autograd.grad(loss, params))
            q = None
            if method == 'usam' or (method == 'cv' and selected):
                probe = copy.deepcopy(base)
                with torch.no_grad():
                    for p, original, gp in zip(probe.parameters(), original_values, g):
                        p.copy_(original + probe_coefficient * gp)
                q = list(torch.autograd.grad(F.cross_entropy(probe(xb), yb), list(probe.parameters())))
            if method == 'usam':
                direction = q
            elif method == 'cv' and selected:
                direction = [2 * qp - gp for gp, qp in zip(g, q)]
            else:
                direction = g
            new_momenta = []
            with torch.no_grad():
                for p, old, gp, v in zip(params, original_values, direction, old_momenta):
                    new_v = momentum * v + gp + decay * old
                    new_momenta.append(new_v)
                    p.copy_(old - eta * new_v)
            return base, new_momenta

        results = {}
        for method, selected in [('sgd', None), ('usam', None), ('cv', False), ('cv', True)]:
            model = copy.deepcopy(initial)
            optimizer = make_optimizer(model)
            attempts = []
            def on_query():
                attempts.append(1)
            result = optimizer_step(model, optimizer, xb, yb, method, eta, gamma,
                                    selected=selected, on_query=on_query)
            reference, new_momenta = independent_reference(method, selected)
            for key, value in model.state_dict().items():
                torch.testing.assert_close(value, reference.state_dict()[key], rtol=2e-12, atol=2e-12)
            for p, v in zip(model.parameters(), new_momenta):
                torch.testing.assert_close(optimizer.state[p]['momentum_buffer'], v, rtol=2e-12, atol=2e-12)
                assert not p.grad.requires_grad
            expected_queries = 2 if method == 'usam' or selected else 1
            assert result['queries'] == len(attempts) == expected_queries
            assert result['examples_queried'] == expected_queries * len(yb)
            assert model.seen_batches == [id(xb)] * expected_queries
            assert model.bn.num_batches_tracked.item() == 1
            assert model.training and model.bn.training
            results[method, selected] = model
        checks.append('SGD, USAM, and both CV branches match independent autograd/manual momentum-decay references')
        checks.append('Same batch, exact query/example counts, detached gradients, and exactly one BN buffer update')

        # The parameter AND momentum-state conditional mean equal USAM at a
        # fixed state, even though full training trajectories need not match.
        for a, b, u in zip(results['cv', False].parameters(), results['cv', True].parameters(),
                           results['usam', None].parameters()):
            torch.testing.assert_close((a + b) / 2, u, rtol=2e-12, atol=2e-12)
        for key in ('bn.running_mean', 'bn.running_var', 'bn.num_batches_tracked'):
            torch.testing.assert_close(results['cv', False].state_dict()[key],
                                       results['cv', True].state_dict()[key], rtol=0, atol=0)
        checks.append('Conditional CV mean matches USAM at a fixed model/momentum state; coin does not alter BN buffers')

        # Exact restore: the optimizer must observe the bitwise original
        # weights, not weights recovered by an add/subtract round trip.
        class CheckingSGD(torch.optim.SGD):
            def __init__(self, parameters, originals):
                super().__init__(parameters, lr=eta, momentum=momentum, weight_decay=decay)
                self.originals = originals
                self.step_calls = 0
            def step(self, closure=None):
                self.step_calls += 1
                for p, original in zip(self.param_groups[0]['params'], self.originals):
                    assert torch.equal(p, original)
                return super().step(closure)
        model = copy.deepcopy(initial)
        optimizer = CheckingSGD(model.parameters(), original_values)
        optimizer_step(model, optimizer, xb, yb, 'cv', eta, gamma, selected=True)
        assert optimizer.step_calls == 1
        checks.append('Probe weights are restored bitwise before exactly one base-optimizer step')

        # An exception in the second query restores parameter and buffer state,
        # counts the attempt, and does not modify momentum or retry.
        model = copy.deepcopy(initial)
        optimizer = make_optimizer(model)
        original_state = {k: v.clone() for k, v in model.state_dict().items()}
        attempts = []
        def fail_second_query():
            attempts.append(1)
            if len(attempts) == 2:
                raise FloatingPointError('Injected second-query failure')
        try:
            optimizer_step(model, optimizer, xb, yb, 'usam', eta, gamma,
                           on_query=fail_second_query)
        except FloatingPointError as error:
            assert 'Injected' in str(error)
        else:
            raise AssertionError('Injected query exception was swallowed.')
        assert len(attempts) == 2
        for key, value in model.state_dict().items():
            assert torch.equal(value, original_state[key])
        for p, old in zip(model.parameters(), old_momenta):
            assert torch.equal(optimizer.state[p]['momentum_buffer'], old)
            assert p.grad is None
        assert model.bn.track_running_stats and model.bn.momentum == initial.bn.momentum
        checks.append('Second-query exception restores weights/BN, preserves momentum, counts attempts, and does not retry')

        # Force a nonfinite second forward, testing the actual BN context's
        # finally path rather than only an early callback failure.
        model = copy.deepcopy(initial)
        optimizer = make_optimizer(model)
        forward_count = []
        def bad_second_forward(module, args, output):
            forward_count.append(1)
            return output * float('nan') if len(forward_count) == 2 else output
        handle = model.register_forward_hook(bad_second_forward)
        try:
            optimizer_step(model, optimizer, xb, yb, 'cv', eta, gamma, selected=True)
        except FloatingPointError:
            pass
        else:
            raise AssertionError('Nonfinite probe was accepted.')
        finally:
            handle.remove()
        assert len(forward_count) == 2
        for key, value in model.state_dict().items():
            assert torch.equal(value, initial.state_dict()[key])
        checks.append('Nonfinite probe backward is rejected and all pre-update weights/BN buffers are restored')

        zero_results = []
        for method, selected in [('sgd', None), ('cv', False), ('cv', True)]:
            model = copy.deepcopy(initial)
            optimizer = make_optimizer(model)
            optimizer_step(model, optimizer, xb, yb, method, eta, 0., selected=selected)
            zero_results.append(model.state_dict())
        for key in zero_results[0]:
            for result in zero_results[1:]:
                torch.testing.assert_close(result[key], zero_results[0][key], rtol=0, atol=0)
        checks.append('At gamma=0 both CV branches exactly match SGD, including momentum/decay and BN')

        model = copy.deepcopy(initial)
        loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(xb, yb), batch_size=4)
        state = {k: v.clone() for k, v in model.state_dict().items()}
        score = evaluate_model(model, loader, 'cpu')
        assert score['count'] == 6 and 0 <= score['accuracy'] <= 1 and math.isfinite(score['loss'])
        assert model.training
        assert all(torch.equal(model.state_dict()[key], value) for key, value in state.items())
        checks.append('Evaluation preserves weights, BN buffers, training mode, and ragged sample counts')

        if real_resnet:
            before_rng = torch.get_rng_state().clone()
            model = build_model(8503, 'cpu')
            assert torch.equal(torch.get_rng_state(), before_rng)
            same = build_model(8503, 'cpu')
            assert all(torch.equal(a, b) for a, b in zip(model.parameters(), same.parameters()))
            del same
            assert model.conv1.kernel_size == (3, 3) and model.conv1.stride == (1, 1)
            assert isinstance(model.maxpool, nn.Identity) and model.fc.out_features == 10
            assert [len(model.layer1), len(model.layer2), len(model.layer3), len(model.layer4)] == [2, 2, 2, 2]
            images = torch.randn(2, 3, 32, 32, generator=generator)
            labels = torch.tensor([1, 7])
            model.train()
            optimizer = torch.optim.SGD(model.parameters(), lr=.01, momentum=.9, weight_decay=5e-4)
            diagnosis = optimizer_step(model, optimizer, images, labels, 'cv', .01, .005, selected=True)
            assert diagnosis['queries'] == 2 and math.isfinite(diagnosis['loss'])
            assert all(m.num_batches_tracked.item() == 1 for m in model.modules() if isinstance(m, nn.BatchNorm2d))
            assert sum(p.numel() for p in model.parameters()) == 11173962
            checks.append('Actual CIFAR ResNet-18: architecture, reproducible isolated initialization, and synthetic two-gradient CV update')
        return {'passed': len(checks), 'checks': checks, 'torch': torch.__version__,
                'device': 'cpu', 'real_resnet_checked': bool(real_resnet)}
    finally:
        torch.set_num_threads(old_threads)




"""Execution-only data checks: synthetic pixels, no downloads or model training."""
from pathlib import Path
import json
import os
import sys
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

data_helpers = sys.modules[__name__]


class WorkerThreadProbe(Dataset):
    def __len__(self):
        return 4

    def __getitem__(self, index):
        return torch.get_num_threads(), os.getpid()


def run_data_checks():
    cfg = {'data_source': 'synthetic', 'train_count': 30, 'val_count': 20,
           'test_count': 20, 'split_seed': 4422, 'subset_seed': 5522,
           'batch_size': 8, 'eval_batch_size': 7, 'num_workers': 0,
           'pin_memory': False, 'prefetch_factor': 2}
    passed = []
    original_cifar = data_helpers.CIFAR10
    calls = []
    def forbidden_cifar(*args, **kwargs):
        calls.append((args, kwargs))
        raise AssertionError('Synthetic checks must not load official CIFAR examples.')
    data_helpers.CIFAR10 = forbidden_cifar
    try:
        data = data_helpers.load_cifar_training('unused-data-directory', cfg)
        data_again = data_helpers.load_cifar_training('unused-data-directory', cfg)
        assert not calls
        assert not data['metadata']['official_test_decoded']
        assert data['metadata'] == data_again['metadata']
        assert np.array_equal(data['images'], data_again['images'])
        assert not np.intersect1d(data['train_indices'], data['validation_indices']).size
        for key, per_class in [('train_indices', 3), ('validation_indices', 2)]:
            assert np.bincount(data['labels'][data[key]], minlength=10).tolist() == [per_class]*10
        assert 'test_indices' not in data
        passed.append('reproducible stratified disjoint training/validation split; test data remain closed')
        test_data = data_helpers.load_cifar_test('unused-data-directory', cfg)
        assert len(test_data['test_indices']) == 20 and not calls
        assert test_data['metadata']['image_data_sha256'] != data['metadata']['image_data_sha256']
        passed.append('test examples exist only through the separate explicit test-loader API')
    finally:
        data_helpers.CIFAR10 = original_cifar

    small = list(data_helpers.EpochBatchSampler(30, 8, 7, 99))
    longer = list(data_helpers.EpochBatchSampler(30, 8, 12, 99))
    assert small == longer[:7]
    assert [len(batch) for batch in small] == [8, 8, 8, 6, 8, 8, 8]
    assert sorted(index for batch in small[:4] for epoch, index in batch) == list(range(30))
    assert all(epoch == 0 for batch in small[:4] for epoch, index in batch)
    assert all(epoch == 1 for batch in small[4:] for epoch, index in batch)
    offset = list(data_helpers.EpochBatchSampler(30, 8, 3, 99, start_update=4))
    assert offset == small[4:]
    passed.append('epoch-keyed sampling retains ragged batches and identical prefixes across horizons')

    dataset = data_helpers.CifarArrayDataset(data['images'], data['labels'],
        data['train_indices'], augment=True, base_seed=987)
    first = dataset[(3, 1)]
    np.random.seed(9912)
    np.random.random(400)
    torch.manual_seed(183)
    torch.randn(100)
    second = dataset[(3, 1)]
    assert torch.equal(first[0], second[0]) and torch.equal(first[1], second[1])
    assert any(not torch.equal(dataset[(0, i)][0], dataset[(1, i)][0]) for i in range(10))
    assert first[0].shape == (3, 32, 32) and first[0].dtype == torch.float32
    passed.append('augmentation is a pure function of seed/epoch/original index, independent of ambient RNG')

    zero = data_helpers.make_train_loader(data, cfg, base_seed=7722, total_updates=7)
    workers_cfg = {**cfg, 'num_workers': 2}
    multi = data_helpers.make_train_loader(data, workers_cfg, base_seed=7722, total_updates=7)
    try:
        zero_batches = list(zero)
        multi_batches = list(multi)
        assert len(zero_batches) == len(multi_batches) == 7
        for (xz, yz), (xm, ym) in zip(zero_batches, multi_batches):
            assert torch.equal(xz, xm) and torch.equal(yz, ym)
        assert [len(y) for x, y in zero_batches] == [8, 8, 8, 6, 8, 8, 8]
    finally:
        data_helpers.close_loader(zero)
        data_helpers.close_loader(multi)
    assert multi._iterator is None
    passed.append('zero workers and two spawn workers produce bitwise-identical augmented batches')

    eval_zero = data_helpers.make_eval_loader(data, cfg)
    eval_multi = data_helpers.make_eval_loader(data, workers_cfg)
    try:
        zero_batches, multi_batches = list(eval_zero), list(eval_multi)
        for (xz, yz), (xm, ym) in zip(zero_batches, multi_batches):
            assert torch.equal(xz, xm) and torch.equal(yz, ym)
        x = torch.cat([batch[0] for batch in zero_batches])
        y = torch.cat([batch[1] for batch in zero_batches])
        reference = torch.from_numpy(data['images'][data['validation_indices']]).permute(0, 3, 1, 2).float() / 255.
        reference = (reference - torch.tensor(data_helpers.CIFAR_MEAN)[None, :, None, None]) / torch.tensor(data_helpers.CIFAR_STD)[None, :, None, None]
        assert torch.equal(x, reference)
        assert torch.equal(y, torch.from_numpy(data['labels'][data['validation_indices']]))
    finally:
        data_helpers.close_loader(eval_zero)
        data_helpers.close_loader(eval_multi)
    passed.append('evaluation has no augmentation, preserves all samples, and is worker-count independent')

    probe = DataLoader(WorkerThreadProbe(), batch_size=2,
                       **data_helpers.loader_options(workers_cfg, 561))
    try:
        rows = list(probe)
        assert all(bool((threads == 1).all()) for threads, pid in rows)
        assert all(bool((pid != os.getpid()).all()) for threads, pid in rows)
    finally:
        data_helpers.close_loader(probe)
    passed.append('worker processes are distinct, have one Torch CPU thread, and shut down cleanly')
    return {'passed': len(passed), 'checks': passed, 'torch': torch.__version__,
            'official_data_downloaded_or_trained': False}
'''

In [19]:
MODULE_SOURCE = '\n\n'.join([DATA_SOURCE, ENGINE_SOURCE, RUNNER_SOURCE, ANALYSIS_SOURCE, CHECK_SOURCE])
IMPLEMENTATION_SHA256 = hashlib.sha256(MODULE_SOURCE.encode()).hexdigest()
CODE_DIR = (WORK_ROOT / 'embedded_code').resolve()
CODE_DIR.mkdir(parents=True, exist_ok=True)
MODULE_NAME = 'cifar_cv_runtime_' + IMPLEMENTATION_SHA256[:20]
MODULE_PATH = CODE_DIR / (MODULE_NAME + '.py')
if MODULE_PATH.exists() and MODULE_PATH.read_text() != MODULE_SOURCE:
    raise RuntimeError('Generated module content does not match its source hash.')
if not MODULE_PATH.exists():
    temporary = MODULE_PATH.with_suffix('.tmp')
    temporary.write_text(MODULE_SOURCE)
    temporary.replace(MODULE_PATH)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
experiment = importlib.import_module(MODULE_NAME)
assert experiment.sha256_file(MODULE_PATH) == IMPLEMENTATION_SHA256
experiment.validate_config(CFG)
print('Generated importable worker module:', MODULE_PATH)

Generated importable worker module: /Users/hooma555/Desktop/Post_Doc/Papers/fashion-mnist-usam-comparison/New/cifar10_usam_cv_sgd/embedded_code/cifar_cv_runtime_45a18df0a7738808b194.py


In [20]:
ENGINE_CHECKS = experiment.run_engine_checks(real_resnet=True)
DATA_CHECKS = experiment.run_data_checks()
print(json.dumps({'engine': ENGINE_CHECKS, 'data': DATA_CHECKS}, indent=2))

{
  "engine": {
    "passed": 9,
    "checks": [
      "SGD, USAM, and both CV branches match independent autograd/manual momentum-decay references",
      "Same batch, exact query/example counts, detached gradients, and exactly one BN buffer update",
      "Conditional CV mean matches USAM at a fixed model/momentum state; coin does not alter BN buffers",
      "Probe weights are restored bitwise before exactly one base-optimizer step",
      "Second-query exception restores weights/BN, preserves momentum, counts attempts, and does not retry",
      "Nonfinite probe backward is rejected and all pre-update weights/BN buffers are restored",
      "At gamma=0 both CV branches exactly match SGD, including momentum/decay and BN",
      "Evaluation preserves weights, BN buffers, training mode, and ragged sample counts",
      "Actual CIFAR ResNet-18: architecture, reproducible isolated initialization, and synthetic two-gradient CV update"
    ],
    "torch": "2.12.0",
    "device": "cpu",
  

## Load training data and inspect the planned work

The smoke profile is deliberately synthetic. The full profile downloads CIFAR-10 if necessary. This is shared setup and is outside per-run time-to-target. The official test split is not decoded here.

In [22]:
data_started = time.perf_counter()
DATA = experiment.load_cifar_training(DATA_DIR, CFG)
DATA_PREPARATION_SECONDS = time.perf_counter() - data_started
TUNE_BUDGET, CHECK_INTERVAL = experiment.phase_budget(DATA, CFG, 'tuning')
BENCH_BUDGET, _ = experiment.phase_budget(DATA, CFG, 'benchmark')
print(json.dumps(DATA['metadata'], indent=2))
print('Shared data preparation seconds:', round(DATA_PREPARATION_SECONDS, 2))
print('Queries per tuning trial:', TUNE_BUDGET, '; per benchmark trial:', BENCH_BUDGET)
print('Validation interval in queries:', CHECK_INTERVAL)
print('Tuning trials:', len(CFG['tuning_seeds']) * len(CFG['lr_grid']) * (1 + 2 * len(CFG['gamma_grid'])))
print('Benchmark trials:', len(CFG['benchmark_seeds']) * (2 + CFG['cv_replicas']))

100.0%


{
  "source": "cifar10",
  "train_count": 45000,
  "validation_count": 5000,
  "official_test_decoded": false,
  "split_seed": 2026102801,
  "subset_seed": 2026102802,
  "train_indices_sha256": "752eceb5f444fc8b9d901e02946c68d44444081267a63eaed76ca7ebde20d2b9",
  "validation_indices_sha256": "230962aaff366bf5fc2c1c0912096b126c83ecac3d30f4e6850138fc96042d7f",
  "image_data_sha256": "47af8b02cfcfb6cdf3dbbd1969ad7f1b70f4413f6f80f437b020ee4e5bc8304a",
  "labels_sha256": "f5cfe00b0f00968c0cc5ff3b1d2de51b10e33efa277c2986f4e0fa63e58c9f4f",
  "normalization": {
    "mean": [
      0.4914,
      0.4822,
      0.4465
    ],
    "std": [
      0.247,
      0.2435,
      0.2616
    ],
    "source": "fixed CIFAR-10 normalization constants; not estimated from test data"
  },
  "archive": {
    "url": "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz",
    "md5": "c58f30108f718f92721af3b95e74349a",
    "training_files": [
      {
        "name": "data_batch_1",
        "md5": "c99cafc152244af7

## Run tuning, freeze selections, and benchmark

All measured jobs run sequentially in a randomized order. A lock prevents overlapping training jobs in this workspace. Saved completed trials are reused without changing their original timings; a changed configuration, runtime, data, or source hash requires a new output directory. Interrupted trials restart from initialization. A crashed kernel can leave a lock: remove it only after confirming that its training process has stopped. Do not run two copies of the benchmark against the same device.

The notebook saves target and final state dictionaries, raw JSON histories, the exact protocol, frozen selections, and summary tables. Full ResNet checkpoints are about 45 MB each; allow space for target and final checkpoints across trials.

In [23]:
TUNING_RECORDS, BENCHMARK_RECORDS, SELECTION = experiment.run_experiment(
    DATA, CFG, OUTPUT_DIR, IMPLEMENTATION_SHA256, lock_root=WORK_ROOT, resume=RESUME)
experiment.atomic_json(OUTPUT_DIR / 'shared_setup.json', {
    'data_preparation_seconds': DATA_PREPARATION_SECONDS,
    'excluded_from_per_run_times': True})
display(pd.DataFrame(SELECTION['candidates']))
print('Frozen selections:')
print(json.dumps(SELECTION['selections'], indent=2))

tuning cv-130d2a1011-s41001-r0 validation: 0.9057999849319458 seconds: 1869.77
tuning cv-2f126e9c06-s41001-r0 validation: 0.9020000100135803 seconds: 1874.58
tuning sgd-cb604ef141-s41001-r0 validation: 0.9297999739646912 seconds: 1770.86
tuning cv-40420d2297-s41001-r0 validation: 0.9124000072479248 seconds: 1872.22
tuning cv-d7195e1681-s41001-r0 validation: 0.8866000175476074 seconds: 1878.86
tuning sgd-19086e9a78-s41001-r0 validation: 0.9205999970436096 seconds: 1778.41
tuning usam-130d2a1011-s41001-r0 validation: 0.9016000032424927 seconds: 1924.79
tuning cv-aa329656d4-s41001-r0 validation: 0.9147999882698059 seconds: 1878.69
tuning usam-40420d2297-s41001-r0 validation: 0.9053999781608582 seconds: 1915.57
tuning usam-aa329656d4-s41001-r0 validation: 0.9016000032424927 seconds: 1916.31
tuning usam-2f126e9c06-s41001-r0 validation: 0.8934000134468079 seconds: 1944.64
tuning sgd-b3ab539b20-s41001-r0 validation: 0.920799970626831 seconds: 1768.85
tuning usam-d7195e1681-s41001-r0 validatio

KeyboardInterrupt: 

## Optional held-out test scoring

Leave `RUN_TEST_SCORING=False` during development. After settings are fixed and every benchmark job has finished, enable it and rerun. Completed training trials are reused. This stage scores both target-confirmation checkpoints and final checkpoints; no model is chosen using test accuracy. An acceptable validation model can still fall short on the test set, so both are reported. Repeated development after inspecting test results makes the evidence exploratory.

In [11]:
if RUN_TEST_SCORING:
    with experiment.exclusive_study_lock(WORK_ROOT):
        SCORED_RECORDS = experiment.score_test_checkpoints(
            BENCHMARK_RECORDS, DATA_DIR, CFG, OUTPUT_DIR, IMPLEMENTATION_SHA256)
else:
    SCORED_RECORDS = BENCHMARK_RECORDS
    print('Official test scoring remains disabled.')

Official test scoring remains disabled.


## Read the answer from the results

Start with target attainment and the primary-target rows. Positive `mean_time_saving_fraction` means CV needed less time than its reference, after averaging any coin replicas within each base seed. A missing time estimate means some planned models never attained the target, not that their time was zero. Check target-model test accuracy, final quality, and tuning costs before making an efficiency claim. The last-iterate comparison uses equal gradient budgets; updates/epochs differ intentionally.

The primary time column includes setup, warmup, validation, and checkpoint I/O. Do not add these components again. `tuning_costs` reports search cost separately so a small training saving is not mistaken for a reduction in the total cost of trying a new optimizer. The plots show measured validation checkpoints without interpolating an unobserved crossing.

In [ ]:
TABLES = experiment.summarize_results([*TUNING_RECORDS, *SCORED_RECORDS], CFG, OUTPUT_DIR)
for table_name in ['failure_table', 'tuning_costs', 'target_summary', 'paired_comparisons', 'final_summary']:
    print(table_name)
    display(TABLES[table_name])
FIGURE = experiment.plot_results(SCORED_RECORDS, CFG, OUTPUT_DIR)
print('Saved results:', OUTPUT_DIR.resolve())

## Checks performed before delivery

The complete notebook executed without cell errors in Jupyter on CPU and Apple MPS with two spawned data-loading workers. Both runs used the actual ResNet-18 and tiny synthetic datasets, exercising stepsize screening, frozen selections, all three methods, checkpoint saving, target attainment and nonattainment, optional test scoring, tables, and plots. Reusing completed trials was also tested on CPU.

Additional checks covered nine optimizer test groups, seven data/worker test groups, nine runner/protocol test groups, and six analysis test groups. They included exact CV directions, single momentum/decay application, parameter and BatchNorm restoration, equal augmented batches with zero versus two workers, failed-gradient accounting, budget boundaries, frozen test-scoring identities, and missing-target handling.

Validation environment: Python 3.14.4, PyTorch 2.12.0, torchvision 0.27.0. CUDA was not available for execution testing. The official CIFAR-10 download was started but stopped because transfer was slow; no real-data training or full benchmark was run. Synthetic smoke results are only execution checks and provide no evidence about accuracy or speedup. The delivered notebook has clean outputs and defaults to the smoke profile.